# 20 - TransMorph-style exploratorio sobre SB013

Este notebook explica y reproduce una prueba de registration deep learning inspirada en TransMorph.

No se utiliza directamente el TransMorph 3D oficial. Se usa una adaptacion ligera 2D porque las bandas de la HSI representan informacion espectral, no profundidad anatomica. La deformacion predicha es espacial 2D y se optimiza por separado para cada pareja H&E-HSI.

## Metodo

- Moving: H&E inicializada mediante el registro afin clasico.
- Fixed: HSI pseudo-RGB y su mascara.
- Entrada: mapas de distancia firmados de las mascaras H&E y HSI.
- Encoder CNN: extrae estructura local.
- Bottleneck Transformer: modela relaciones espaciales globales.
- Decoder CNN: predice un campo denso de deformacion 2D.
- Perdida: similitud de mapas de distancia + Dice de mascaras + suavidad.

El modelo es **instance-specific**: no aprende un modelo universal con seis sujetos, sino que optimiza una red para la pareja seleccionada.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
METHOD_DIR = ROOT / 'Registration' / 'DeepLearning' / 'Tecnicas_registration' / '02_transmorph'
SCRIPT_DIR = METHOD_DIR / 'scripts'
SCRIPT = SCRIPT_DIR / 'run_transmorph_instance.py'
OUTPUT_TAG = 'tm_s100_d008'
SUBJECT = 'SB013'
OUTPUT_DIR = METHOD_DIR / 'outputs' / f'outputs_transmorph_instance_{OUTPUT_TAG}_{SUBJECT}'

if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

print('Script:', SCRIPT)
print('Resultado existente:', OUTPUT_DIR)

In [ ]:
# Arquitectura utilizada en el experimento.
from run_transmorph_instance import TransMorphLite2D

model = TransMorphLite2D(
    input_size=192,
    max_disp_norm=0.08,
    transformer_depth=2,
    transformer_heads=4,
)
num_params = sum(parameter.numel() for parameter in model.parameters())
print(f'Parametros entrenables: {num_params:,}')
print(model)

## Ejecucion opcional

El resultado seleccionado ya existe. Deja `RUN_TRAINING = False` para revisar los resultados sin volver a entrenar. Cambialo a `True` solo si quieres repetir SB013 desde cero; se guardara con la etiqueta `notebook_demo` para no sobrescribir la grid final.

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    command = [
        sys.executable,
        str(SCRIPT),
        '--subject', SUBJECT,
        '--iterations', '140',
        '--size', '192',
        '--lr', '0.0015',
        '--smooth-weight', '1.00',
        '--dice-weight', '0.45',
        '--max-disp-norm', '0.08',
        '--transformer-depth', '2',
        '--transformer-heads', '4',
        '--output-tag', 'notebook_demo',
    ]
    result = subprocess.run(command, cwd=str(ROOT), text=True, capture_output=True)
    print(result.stdout[-5000:])
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError('Fallo el entrenamiento TransMorph-style de demostracion')

In [ ]:
summary_path = OUTPUT_DIR / f'{SUBJECT}_transmorph_instance_summary.json'
with open(summary_path, encoding='utf-8') as handle:
    summary = json.load(handle)

keys = [
    'subject', 'method', 'device', 'input_size', 'iterations',
    'initial_dice', 'final_dice', 'flow_p95_px',
    'jacobian_det_negative_fraction',
]
display(pd.DataFrame([{key: summary[key] for key in keys}]).T.rename(columns={0: 'value'}))

In [ ]:
summary_image = OUTPUT_DIR / f'{SUBJECT}_transmorph_instance_summary.png'
plt.figure(figsize=(16, 6))
plt.imshow(Image.open(summary_image))
plt.axis('off')
plt.title('SB013: entrada afin, resultado TransMorph-style y curva de optimizacion')
plt.tight_layout()

## Interpretacion

SB013 es uno de los casos mas estables. La configuracion conservadora mejora el Dice del baseline clasico de aproximadamente 0.984 a 0.996 y mantiene indicadores de deformacion relativamente controlados.

Este buen resultado no demuestra generalizacion. Demuestra que una arquitectura hibrida CNN-Transformer puede optimizar una deformacion util para una pareja concreta H&E-HSI.